# Heart Disease Classification Project

## Predictiing Heart disease using machine learning

This notebook looks into using various Python-based machine learning and data science libraries in an attempt to build machine learning models capable of predicting whether or not someone has heart disease based on some medical attributes.

We're going to take the following approach
1. Problem Definition
2. Data
3. Evaluation
4. Features
5. Modelling
6. Experimentation

## 1. Problem Definition

> Given clinical parameters about a patient , can we predict whether or not they have heart disease?

## 2. Data

The original data came from the UCI Machine Learning Repository:https://archive.ics.uci.edu/dataset/45/heart+disease

## 3. Evaluation

> If we can reach 95% accuracy at predicting whether or not a patient has heart disease during the proof of concept , we'll pursue the project



## 4. Features
      1. (age)       
      2. (sex)       
      3. (cp)        
      4. (trestbps)  
      5. (chol)      
      6. (fbs)       
      7. (restecg)   
      8. (thalach)   
      9. (exang)     
      10. (oldpeak)   
      11. (slope)     
      12. (ca)        
      13. (thal)      
      14. (target) 


## Prepare the tools

We're going to use pandas, Matplotlib,Numpy for data analysis/manipulation.

In [6]:
#Import the tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay

%matplotlib inline

## Load Data

In [7]:
data=pd.read_csv("heart-disease (2).csv")

FileNotFoundError: [Errno 2] No such file or directory: 'heart-disease (2).csv'

## Data exploration(Exploratory data analysis or EDA)

The goal here is to find out more about the data and become a subject matter export on the dataset you are working with.

1. What question(s) are you trying to solve?
2. What kind of data do we have and how we treat different types?
3. What's missing from the data and how do you deal with it?
4. Where are the outliers and why should you care about them?'
5. How can you add , change or remove features to get out more of your data?

In [ ]:
data.head()

In [ ]:
data["target"].value_counts()

In [ ]:
data["target"].value_counts().plot(kind="bar",
                                   color=["red","blue"],
                                   figsize=(8,5));
plt.title("Target values count",fontweight="bold")
plt.tight_layout()
plt.ylabel("Counts")
plt.yticks(np.arange(0,180,10));

In [ ]:
#Are there any missing values?
data.isna().sum()

### Heart Disease Frequency according to Sex

In [ ]:
data["sex"].value_counts()

In [ ]:
#Compare target column with sex column
pd.crosstab(data["target"],data["sex"])

In [ ]:
pd.crosstab(data["target"],data["sex"]).plot(kind="bar",
                                             color=["red","blue"],
                                             figsize=(8,5))

plt.title("Heart Disease Frequency for Sex",fontweight="bold")
plt.ylabel("Amount")
plt.tight_layout()
plt.yticks(np.arange(0,150,10))
plt.legend(["Female","Male"]);

### Age vs Max Heart Rate for Heart Disease`

In [ ]:
#Create another figure
plt.figure(figsize=(10,6))

#Scatter with positive examples
plt.scatter(data.age[data.target==1],
            data.thalach[data.target==1],
            c="blue");

plt.scatter(data.age[data.target==0],
            data.thalach[data.target==0],
            c="red");

In [ ]:
# Check the distribution of the age column with a histogram
data.age.hist(bins=50);

### Heart Disease Frequency per Cholesterol Levels

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2)
fig.suptitle("Heart Disease depending on cholesterol",fontweight="bold");
fig.set_size_inches(12, 5)


ax1.hist(data[data["target"]==0]["chol"],bins=40);
ax1.set_title("No Heart Disease")

ax2.hist(data[data["target"]==1]["chol"],bins=40,color="red")
ax2.set_title("Heart Disease");
plt.tight_layout()

### Heart Disease Frequency per Chest Pain Type

In [ ]:
pd.crosstab(data["cp"],data["target"])

In [ ]:
#Make the crosstab more visual
pd.crosstab(data["cp"],data["target"]).plot(kind="bar")
plt.legend(["Yes", "No"])
plt.title("Heart Disease Frequency per Chest Pain")
plt.xlabel("Chest Pain Type");

In [ ]:
# Make a correlation matrix 
data.corr()

In [ ]:
corr_matrix=data.corr()
fig, ax=plt.subplots(figsize=(10,10))

ax=sns.heatmap(corr_matrix,
               annot=True,
               fmt=".2f")

## 5. Modelling

In [ ]:
# Split the data into X & y

X=data.drop("target", axis=1)
y=data["target"]

In [ ]:
from sklearn.model_selection import train_test_split

np.random.seed(42)

#Split into train and test splits
X_train, X_test, y_train, y_test= train_test_split(X, y , test_size=0.2)

#Scaling
scaler= StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.fit_transform(X_test)

Now we've got our data split in train and test splits it's time to train 
a Machine learning model

Three different Machine Learning Models are used: 
1. Logistic Regression
2. K- Nearest Neighbors Classifier
3. Random Forest Classifier

In [ ]:
# Put models in adictionary

models={"Logistic Regression": LogisticRegression(max_iter=100),
        "KNN":KNeighborsClassifier(),
        "Random Forest": RandomForestClassifier()}

#Create a function to fit and score models

def fit_and_score(models, X_train, X_test, y_train, y_test):
    #Set random seed
    np.random.seed(42)
    #Make a dictionary to keep model scores
    model_score={}
    for name,model in models.items():
        model.fit(X_train, y_train)
        model_score[name]=model.score(X_test, y_test)
    return model_score

results=fit_and_score(models, X_train, X_test, y_train, y_test)
results

Let's look at the following:
* Hyperparameter tuning
* Feature importance
* Confusion matrix
* Cross-validation
* Precision
* Recall
* F1 score
* Classification report
* ROC curve
* Area under the curve

### Hyperparameter Tuning with RandomizedSearchCV

We're going to tune:
* LogisticRegression()
* RandomForestClassifier()
using RandomizedSearchCV

In [ ]:
#Create a hyperparameter grid for Logistic Regression

log_reg_grid={"C": np.logspace(-4,4,20),
              "penalty":["l1","l2","elasticnet","none"],
              "solver":["liblinear","newton-cg","lbfgs","sag"],
              }

#Create a hyperparameter grid for RandomForestClassifier

rf_grid={"n_estimators": np.arange(10,1000,50),
         "max_depth": [None,10,20],
         "min_samples_split": [2,5],
         "min_samples_leaf": [1,2],
         "bootstrap":[True, False]}

In [ ]:
#Tune Logistic Regression

np.random.seed(42)

#Setup random hyperparameter search for LogisticRegression
rs_log_reg=RandomizedSearchCV(LogisticRegression(),
                              param_distributions=log_reg_grid,
                              cv=5,
                              n_iter=20,
                              verbose=2)

In [ ]:
#Fit random hyperparameter search for LogisticRegression
rs_log_reg.fit(X_train, y_train)

In [ ]:
rs_log_reg.best_params_

In [ ]:
rs_log_reg.score(X_test, y_test)

In [ ]:
#Tune RandomForestClassifier

np.random.seed(42)

#Setup random hyperparameter search for RandomForest
rs_rf=RandomizedSearchCV(RandomForestClassifier(),
                           param_distributions=rf_grid,
                           cv=5,
                           n_iter=20,
                           verbose=2)

In [ ]:
#Fit RandomizedSearchCV for RandomForest

rs_rf.fit(X_train, y_train)

In [ ]:
#Find the best hyperparameters

rs_rf.best_params_

In [ ]:
# Evaluate the randomized search RandomForestClassifier model
rs_rf.score(X_test,y_test)

## Hyperparameter Tuning with GridSearchCV

In [ ]:
#Different hyperparameters for our LogisticRegression model
log_reg_grid={"C":np.logspace(-4,4,30),
              "solver":["liblinear"]}

#Setup grid hyperparameter search for LogisticRegression
gs_log_reg= GridSearchCV(LogisticRegression(),
                        param_grid=log_reg_grid,
                        cv=5,
                        verbose=2)

In [ ]:
#Fit the model
gs_log_reg.fit(X_train, y_train)


In [ ]:
#Check the best hyperparams
gs_log_reg.best_params_

In [ ]:
#Evaluate the model
gs_log_reg.score(X_test, y_test)

In [ ]:
# Different hyperparameters for our RandomForest Classifier
random_forest_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

gs_random_forest= GridSearchCV(RandomForestClassifier(),
                               param_grid=random_forest_grid,
                               cv=5,
                               verbose=2)

In [ ]:
# Fit the model
gs_random_forest.fit(X_train, y_train)

In [ ]:
gs_random_forest.best_params_

In [ ]:
gs_random_forest.score(X_test, y_test)

## Evaluating our tuned machine learning classifier, beyond accuracy

* ROC curve and AUC score
* Confusion matrix
* Classification report
* Precision
* Recall
* F1-score

... and it would be great if cross-validation was used where possible

In [ ]:
 #Make predictions with tuned model
y_preds=gs_log_reg.predict(X_test)
print("Grid Search CV Logistic Regression results:",y_preds)

y_preds2=gs_random_forest.predict(X_test)
print("Grid Search CV Random Forest results:",y_preds)


In [ ]:
#Import ROC curve from sklearn.metrics
from sklearn.metrics import RocCurveDisplay,roc_curve,auc

#Plot ROC curve and calculate AUC
y_scores=gs_log_reg.predict_proba(X_test)[:,1]
fpr, tpr,_=roc_curve(y_test,y_scores)
roc_auc=auc(fpr,tpr)

y2_scores=gs_random_forest.predict_proba(X_test)[:,1]
fpr2,tpr2,_=roc_curve(y_test, y2_scores)
roc_auc2=auc(fpr2,tpr2)

plt.figure()
plt.plot(fpr,tpr,label=f'Logistic Regression (AUC={roc_auc:.2f})')
plt.plot(fpr2,tpr2,label=f'Random Forest (AUC={roc_auc2:.2f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random Guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve and AUC for Logistic Regression/Random Forest")
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test,y_preds))

In [ ]:

fig, ax =plt.subplots(1,2,figsize=(3,3))
sns.heatmap(confusion_matrix(y_test, y_preds),
               annot=True,
               ax=ax[0])
sns.heatmap(confusion_matrix(y_test, y_preds2),
               annot=True,
               ax=ax[1])
fig.set_size_inches(12, 5)
ax[0].set_xlabel("True label")
ax[0].set_ylabel("Predicted label")

ax[1].set_xlabel("True label")
ax[1].set_ylabel("Predicted label")

ax[0].set_title("Logistic Regression Heatmap")
ax[1].set_title("Random Forest  Heatmap")
plt.show();

In [ ]:
print(classification_report(y_test, y_preds))

### Calculate evaluation metrics using cross-validation

Calculate precision, recall and f1_score using cross_val_score()

In [ ]:
#Check best parameters

gs_log_reg.best_params_

In [ ]:
#Create a new classifier with best parameters

clf=LogisticRegression(C= 0.01610262027560939, solver= 'liblinear')

In [ ]:
#Cross validated accuracy

cross= cross_val_score(clf,
                            X,
                            y,
                            cv=5,
                            scoring="accuracy")
cross=np.mean(cross)

In [ ]:
cross2= cross_val_score(clf,
                        X,
                        y,
                        cv=5,
                         scoring="recall")
cross2=np.mean(cross2)

In [ ]:
cross3= cross_val_score(clf,
                        X,
                        y,
                        cv=5,
                         scoring="f1")
cross3=np.mean(cross3)

In [ ]:
# Visualize cross-validated metrics
metrics = ["Accuracy", "Recall", "F1-score"]
scores = [cross, cross2, cross3] # Αν είναι δεκαδικοί (π.χ. 0.85), πολλαπλασιάστε με 100

fig, ax = plt.subplots(figsize=(6, 7))

# Αποθήκευση του γραφήματος στη μεταβλητή bars
bars = plt.barh(metrics, scores, color='mediumseagreen', ec='k', lw=1)

# Προσθήκη των ετικετών με το σύμβολο %
ax.bar_label(bars, fmt='%.3f%%', padding=3)

plt.ylabel("Score")
plt.xlabel("Metric")
plt.title("Model Performance (Cross Validation)")
fig.set_size_inches(12.5, 6.5)
plt.show()

In [ ]:
import pickle

with open("gs_log_reg.pkl","wb") as file:
    pickle.dump(gs_log_reg,file)

### Feature Importance

Which features contribute the most to the outcomes of our model?

In [ ]:
clf = LogisticRegression(C=0.01610262027560939, solver= 'liblinear')


clf.fit(X_train, y_train)
for i,coef in enumerate(clf.coef_[0]):
    print(f"Feature {i} has coefficent: {coef}")

In [ ]:
feature_importance_df=pd.DataFrame(dict(zip(X.columns,clf.coef_[0])),index=[0])

In [ ]:
#Visualize feature Importance
feature_importance_df.T.sort_values(by=0, ascending=True).plot(kind="bar")
plt.title("Logistic Regression Feature Importance");

### Feature Importance Analysis (Logistic Regression)

The feature importance analysis was conducted using the coefficients derived from the Logistic Regression model. The magnitude of each coefficient indicates the strength of the feature’s influence on the prediction, while the sign (positive or negative) reflects the direction of this influence.

From the analysis, it was observed that chest pain type (cp) is the most influential feature, showing the highest positive coefficient, indicating a strong association with the presence of heart disease. Similarly, maximum heart rate achieved (thalach) and slope of the ST segment (slope) also exhibit significant positive contributions, suggesting that higher values of these features increase the predicted probability of heart disease.

On the other hand, features such as oldpeak, number of major vessels (ca), thalassemia (thal), and exercise-induced angina (exang) display strong negative coefficients. This indicates that these variables are associated with a decreased probability of the target class, according to the model. Additionally, sex also shows a noticeable negative contribution.

Some features, including fasting blood sugar (fbs), cholesterol (chol), and age, appear to have relatively small coefficients, suggesting a limited impact on the model’s predictions.

Overall, the model highlights a subset of features that play a dominant role in classification, while others contribute minimally. This insight can be useful for feature selection, model simplification, and interpretability.

In [ ]:
#Feature Importance (Random Forest Classifier)

gs_random_forest.best_params_

In [ ]:
clf_2=RandomForestClassifier(bootstrap= True,
 max_depth=20,
 max_features= 'sqrt',
 min_samples_leaf= 2,
 min_samples_split= 2,
 n_estimators=100)
clf_2.fit(X_train, y_train)

In [ ]:
clf_2.feature_importances_

In [ ]:
feature_importance_rf=pd.DataFrame(dict(zip(X.columns,clf_2.feature_importances_)),index=[0])

In [ ]:
feature_importance_rf.T.sort_values(by=0, ascending = True).plot(kind="bar",legend=False)
plt.title("Random Forest Classifier Feature Importance");